In [3]:
import duckdb, numpy as np, torch, torch.nn as nn
from sklearn.metrics import roc_auc_score

con = duckdb.connect()
df = con.execute("""
    SELECT seq_id, fold, pos, qid, response
    FROM '../data/processed/interactions.parquet'
    ORDER BY seq_id, pos
""").df()

qids = np.sort(df.qid.unique())
qmap = {q: i + 1 for i, q in enumerate(qids)}   # 0은 패딩
df["q"] = df.qid.map(qmap).astype("int64")
NQ = len(qids)

MAXLEN = 200
seqs = []
for sid, g in df.groupby("seq_id", sort=False):
    q, r, f = g.q.values, g.response.values, int(g.fold.iloc[0])
    for s in range(0, len(q), MAXLEN):
        if len(q[s:s+MAXLEN]) >= 3:
            seqs.append((f, q[s:s+MAXLEN], r[s:s+MAXLEN]))

print(NQ, len(seqs))

7618 30965


In [4]:
def make_batch(items):
    L = max(len(x[1]) for x in items)
    Q = np.zeros((len(items), L), dtype=np.int64)
    R = np.zeros((len(items), L), dtype=np.int64)
    for i, (_, q, r) in enumerate(items):
        Q[i, :len(q)] = q
        R[i, :len(r)] = r
    return torch.from_numpy(Q), torch.from_numpy(R)

class DKT(nn.Module):
    def __init__(self, nq, h=128):
        super().__init__()
        self.emb = nn.Embedding(2 * nq + 1, h, padding_idx=0)
        self.rnn = nn.LSTM(h, h, batch_first=True)
        self.out = nn.Linear(h, nq + 1)

    def forward(self, q, r):
        tok = torch.where(q > 0, (q - 1) * 2 + r + 1, torch.zeros_like(q))
        h, _ = self.rnn(self.emb(tok[:, :-1]))
        logits = self.out(h)
        return logits.gather(2, q[:, 1:].unsqueeze(-1)).squeeze(-1)

dev = "cuda"
VAL = 0
train = [s for s in seqs if s[0] != VAL]
val = [s for s in seqs if s[0] == VAL]
model = DKT(NQ).to(dev)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
lossf = nn.BCEWithLogitsLoss(reduction="none")

In [5]:
def run_eval():
    model.eval(); P, Y = [], []
    with torch.no_grad():
        for i in range(0, len(val), 256):
            q, r = [t.to(dev) for t in make_batch(val[i:i+256])]
            m = q[:, 1:] > 0
            p = torch.sigmoid(model(q, r))
            P.append(p[m].cpu().numpy()); Y.append(r[:, 1:][m].cpu().numpy())
    return roc_auc_score(np.concatenate(Y), np.concatenate(P))

EPOCHS = 5
for ep in range(EPOCHS):
    model.train()
    idx = np.random.permutation(len(train)); tot = 0
    for i in range(0, len(idx), 128):
        q, r = [t.to(dev) for t in make_batch([train[j] for j in idx[i:i+128]])]
        m = (q[:, 1:] > 0).float()
        loss = (lossf(model(q, r), r[:, 1:].float()) * m).sum() / m.sum()
        opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
    print(ep, round(tot / (len(idx) / 128), 4), "val AUC", round(run_eval(), 4))

0 0.4719 val AUC 0.7847
1 0.4038 val AUC 0.807
2 0.39 val AUC 0.8126
3 0.3828 val AUC 0.8152
4 0.3776 val AUC 0.816


In [6]:
print(next(model.parameters()).device)

cuda:0


In [7]:
import json, os
os.makedirs("../models", exist_ok=True)
torch.save(model.state_dict(), "../models/dkt_fold0.pt")
json.dump({str(k): v for k, v in qmap.items()}, open("../models/qmap.json", "w"))

In [8]:
import pandas as pd
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
    f1_score, precision_score, recall_score, log_loss, brier_score_loss, confusion_matrix)

model.eval(); rows = []
with torch.no_grad():
    for i in range(0, len(val), 256):
        batch = val[i:i+256]
        q, r = [t.to(dev) for t in make_batch(batch)]
        p = torch.sigmoid(model(q, r)).cpu().numpy()
        qn, rn = q[:, 1:].cpu().numpy(), r[:, 1:].cpu().numpy()
        for b in range(len(batch)):
            n = len(batch[b][1]) - 1
            rows.append(pd.DataFrame({"seq": i + b, "pos": np.arange(1, n + 1),
                "q": qn[b, :n], "y": rn[b, :n], "p": p[b, :n]}))
ev = pd.concat(rows, ignore_index=True)

# 1) 전체 지표
y, p = ev.y.values, ev.p.values
pred = (p >= 0.5).astype(int)
print("행 수:", len(ev), "| 정답률:", round(y.mean(), 4))
print("AUC:", round(roc_auc_score(y, p), 4), "| PR-AUC:", round(average_precision_score(y, p), 4))
print("Accuracy:", round(accuracy_score(y, pred), 4), "| 다수클래스 기준:", round(max(y.mean(), 1 - y.mean()), 4))
print("Precision/Recall/F1 (정답=1):", round(precision_score(y, pred), 4),
      round(recall_score(y, pred), 4), round(f1_score(y, pred), 4))
print("오답(0) 기준 Precision/Recall/F1:", round(precision_score(y, pred, pos_label=0), 4),
      round(recall_score(y, pred, pos_label=0), 4), round(f1_score(y, pred, pos_label=0), 4))
print("LogLoss:", round(log_loss(y, p), 4), "| Brier:", round(brier_score_loss(y, p), 4))
print("혼동행렬 [[TN FP],[FN TP]]:\n", confusion_matrix(y, pred))

# 2) 보정(캘리브레이션)과 ECE
ev["bin"] = pd.cut(ev.p, np.linspace(0, 1, 11), include_lowest=True)
cal = ev.groupby("bin", observed=True).agg(n=("y", "size"), mean_pred=("p", "mean"), actual=("y", "mean"))
print(cal.round(3))
print("ECE:", round((cal.n / cal.n.sum() * (cal.mean_pred - cal.actual).abs()).sum(), 4))

# 3) 풀이 위치별 AUC (이력이 쌓일수록 좋아지는가)
ev["pos_bin"] = pd.cut(ev.pos, [0, 5, 10, 20, 50, 100, 200])
def auc_safe(g): return roc_auc_score(g.y, g.p) if g.y.nunique() == 2 else np.nan
print(ev.groupby("pos_bin", observed=True).apply(auc_safe).round(4))

# 4) 문항 학습 빈도별 AUC (희귀 문항에서 약한가)
cnt = pd.Series(np.concatenate([s[1] for s in train])).value_counts()
ev["freq"] = ev.q.map(cnt).fillna(0)
ev["freq_bin"] = pd.cut(ev.freq, [-1, 0, 50, 200, 1000, 1e9])
print(ev.groupby("freq_bin", observed=True).apply(auc_safe).round(4))

# 5) 시퀀스(학생)별 AUC 분포
sa = ev.groupby("seq").apply(auc_safe).dropna()
print("시퀀스별 AUC 평균/중앙값/10%/90%:", sa.mean().round(4), sa.median().round(4),
      sa.quantile(0.1).round(4), sa.quantile(0.9).round(4))

# 6) 기준선 A를 같은 행에서 재계산 (공정 비교)
tq = pd.concat([pd.DataFrame({"q": s[1], "y": s[2]}) for s in train])
acc = tq.groupby("q").y.mean()
pa = ev.q.map(acc).fillna(tq.y.mean())
print("같은 행 기준 A의 AUC:", round(roc_auc_score(y, pa), 4), "| DKT:", round(roc_auc_score(y, p), 4))

행 수: 885091 | 정답률: 0.7945
AUC: 0.816 | PR-AUC: 0.9395
Accuracy: 0.8286 | 다수클래스 기준: 0.7945
Precision/Recall/F1 (정답=1): 0.8471 0.9568 0.8987
오답(0) 기준 Precision/Recall/F1: 0.666 0.3326 0.4436
LogLoss: 0.3918 | Brier: 0.1235
혼동행렬 [[TN FP],[FN TP]]:
 [[ 60503 121401]
 [ 30347 672840]]
                    n  mean_pred  actual
bin                                     
(-0.001, 0.1]    3882      0.057   0.040
(0.1, 0.2]      10281      0.156   0.134
(0.2, 0.3]      17325      0.253   0.248
(0.3, 0.4]      24687      0.353   0.353
(0.4, 0.5]      34675      0.453   0.456
(0.5, 0.6]      48565      0.553   0.559
(0.6, 0.7]      70468      0.653   0.654
(0.7, 0.8]     107748      0.754   0.746
(0.8, 0.9]     185799      0.855   0.845
(0.9, 1.0]     381661      0.955   0.949
ECE: 0.0067
pos_bin
(0, 5]        0.7776
(5, 10]       0.7991
(10, 20]      0.8088
(20, 50]      0.8074
(50, 100]     0.8168
(100, 200]    0.8245
dtype: float64
freq_bin
(-1.0, 0.0]               0.6298
(0.0, 50.0]             